
# 練習問題: omp simd reduction で内積をSIMD化する

## 目標

総和をとるループ (リダクション) を `#pragma omp simd reduction(+:s)` (Fortran では `!$omp simd reduction(+:s)`) でSIMD化する.
00 の saxpy のような要素ごと独立の演算 (elementwise) と違い, 内積は1つの変数 `s` に値を足し込んでいくため, そのままでは各反復に依存関係があり自動ベクトル化されにくい. `reduction` 節で「部分和を複数レーンに分けて最後に合計してよい」とコンパイラに伝えるのがポイントで, 00 より一歩進んだ内容である.

## 課題

`dot_simd.cpp` (または `dot_simd.f90`) は内積 `s = Σ x[i]*y[i]` を計算する関数 `dot` を持つ.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, 総和ループがSIMD化されるようにせよ.

- C++: `for` ループの直前に `#pragma omp simd reduction(+:s)` を1行加える.
- Fortran: `do` ループの直前に `!$omp simd reduction(+:s)` を1行加える.

それ以外のコードを変更する必要はない.

## コンパイルと実行

`#pragma omp simd` を解釈させるには `-mp=multicore` が必要である (SIMD命令を使うだけなので1スレッドで動作する).

```
# C++
nvc++ -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo dot_simd.cpp -o dot_simd_cpp.exe
./dot_simd_cpp.exe

# Fortran
nvfortran -fast -mp=multicore -Mkeepasm -Minfo -Mneginfo dot_simd.f90 -o dot_simd_f90.exe
./dot_simd_f90.exe
```

`x[i]=1`, `y[i]=2` としているので, 内積は理論値 `2*n` になり, `OK` と表示されれば正しい.

生成されたアセンブリ (`dot_simd.s`) を確認すると, 積和のSIMD命令 (`vfmadd...pd` など packed double 命令) と, 最後に複数レーンの部分和をまとめる水平加算 (horizontal add) が現れる.

## 期待される結果

- `OK: s=200000000.0 (= 2*n)` のように, 内積が理論値 `2*n` に一致する.
- `dot_simd.s` の中に `pd` 付きの packed double 命令が現れ, `-Minfo` に "Generated vector ..." が出る.
- Fortran では組込み関数 `dot_product(x, y)` を使っても同様にベクトル化される (余裕があれば比べてみよ).



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ dot_simd.cpp
#include <cstdio>
#include <cstdlib>

/* 内積 s = Σ x[i]*y[i] を n 要素について計算する */
double dot(long n, double * x, double * y) {
  double s = 0.0;
  // TODO: 内積の総和ループを simd reduction でSIMD化せよ (下の for の直前に1行追加).
  for (long i = 0; i < n; i++) {
    s += x[i] * y[i];
  }
  return s;
}

int main(int argc, char ** argv) {
  long n = (argc > 1 ? atol(argv[1]) : 100L * 1000 * 1000);
  double * x = (double *)malloc(sizeof(double) * n);
  double * y = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) { x[i] = 1.0; y[i] = 2.0; }

  double s = dot(n, x, y);

  /* x[i]=1, y[i]=2 なので理論値は 2*n */
  double expected = 2.0 * (double)n;
  if (s == expected) printf("OK: s=%.1f (= 2*n)\n", s);
  else               printf("NG: s=%.1f, expected=%.1f\n", s, expected);
  free(x); free(y);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore dot_simd.cpp -o dot_simd_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./dot_simd_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ dot_simd.f90
! 内積 s = Σ x(i)*y(i) を n 要素について計算する.
! 注: Fortran では組込み関数 dot_product(x, y) も同様にSIMD化される.
function dot(n, x, y) result(s)
  implicit none
  integer(8), intent(in) :: n
  real(8), intent(in) :: x(n), y(n)
  real(8) :: s
  integer(8) :: i
  s = 0.0d0
  ! TODO: 内積の総和ループを simd reduction でSIMD化せよ (下の do の直前に1行追加).
  do i = 1, n
     s = s + x(i) * y(i)
  end do
end function dot

program dot_simd
  implicit none
  character(len=32) :: arg
  integer(8) :: n, i
  real(8), allocatable :: x(:), y(:)
  real(8) :: s, expected, dot
  n = 100_8 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(x(n), y(n))
  do i = 1, n
     x(i) = 1.0d0; y(i) = 2.0d0
  end do

  s = dot(n, x, y)

  ! x(i)=1, y(i)=2 なので理論値は 2*n
  expected = 2.0d0 * real(n, 8)
  if (s == expected) then
     print "(a,f0.1,a)", "OK: s=", s, " (= 2*n)"
  else
     print "(a,f0.1,a,f0.1)", "NG: s=", s, ", expected=", expected
  end if
end program dot_simd

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore dot_simd.f90 -o dot_simd_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./dot_simd_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:dot_simd.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:dot_simd.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:dot_simd.cpp}